In [1]:
# Check that the class reproduces our results 

%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
from src.classifier import TransactionClassifier
from src.evaluation import evaluate

df = pd.read_csv("../data/processed/transactions_clean.csv")
realistic = pd.read_csv("../data/test_realistic.csv")

for name, use_embeddings in [("Baseline", False), ("Hybrid", True)]:
    classifier = TransactionClassifier(use_embeddings=use_embeddings).fit(df["label"], df["category"])
    predictions = classifier.predict(realistic["label"])
    predicted = [category for category, _ in predictions]
    evaluate(realistic["category"], predicted, f"{name} (realistic set)")

Baseline (realistic set)         accuracy: 73.3% | macro F1: 73.4%


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Hybrid (realistic set)           accuracy: 71.7% | macro F1: 71.0%


In [2]:
# Out-of-fold predictions for both models 

from sklearn.model_selection import StratifiedGroupKFold
from src.evaluation import brand_of, split_by_brand

train, test = split_by_brand(df)


def out_of_fold(use_embeddings):
    groups = train["merchant"].apply(brand_of)
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    rows = []
    for train_idx, val_idx in splitter.split(train, train["category"], groups):
        fold_train, fold_val = train.iloc[train_idx], train.iloc[val_idx]
        classifier = TransactionClassifier(use_embeddings=use_embeddings)
        classifier.fit(fold_train["label"], fold_train["category"])
        predictions = classifier.predict(fold_val["label"])
        for (category, confidence), label, truth in zip(predictions, fold_val["label"], fold_val["category"]):
            rows.append({
                "label": label,
                "category": truth,
                "predicted": category,
                "confidence": confidence,
                "correct": category == truth,
            })
    return pd.DataFrame(rows)


oof = {
    "Baseline": out_of_fold(use_embeddings=False),
    "Hybrid": out_of_fold(use_embeddings=True),
}
for name, predictions in oof.items():
    print(f"{name:<10} out-of-fold accuracy: {predictions['correct'].mean():.1%}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Baseline   out-of-fold accuracy: 51.3%
Hybrid     out-of-fold accuracy: 57.4%


In [3]:
# The threshold trade-off

rows = []
for name, predictions in oof.items():
    for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        confident = predictions["confidence"] >= threshold
        rows.append({
            "model": name,
            "threshold": threshold,
            "sent to Claude": 1 - confident.mean(),
            "accuracy on kept": predictions.loc[confident, "correct"].mean(),
        })

table = pd.DataFrame(rows).pivot(index="threshold", columns="model")
table.style.format("{:.0%}")

In [4]:
# A first batch 

from dotenv import load_dotenv
from src.classifier import ask_claude

load_dotenv("../.env")

examples = [
    "CB CINEMA UGC LES HALLES 12/03",
    "PRLV SEPA BASIC FIT FR12ZZZ123456",
    "VIR SEPA RECU DE DUPONT SAS",
    "CB DECATHLON ROUEN 05/04",
    "VIR INST VERS M LEROY PAUL",
    "CB PICARD SURGELES 22/03",
]

for label, category in zip(examples, ask_claude(examples)):
    print(f"{label:<38} -> {category}")

CB CINEMA UGC LES HALLES 12/03         -> Leisure & Travel
PRLV SEPA BASIC FIT FR12ZZZ123456      -> Leisure & Travel
VIR SEPA RECU DE DUPONT SAS            -> Income
CB DECATHLON ROUEN 05/04               -> Shopping
VIR INST VERS M LEROY PAUL             -> Transfers
CB PICARD SURGELES 22/03               -> Groceries


In [5]:
# Ask claude about 200 uncertain labels

from pathlib import Path

sample_path = Path("../data/processed/claude_sample.csv")

if sample_path.exists():
    sample = pd.read_csv(sample_path)
    print("Loaded saved Claude answers (no API calls).")
else:
    hybrid_oof = oof["Hybrid"]
    sample = hybrid_oof[hybrid_oof["confidence"] < 0.9].sample(200, random_state=42).reset_index(drop=True)

    BATCH_SIZE = 25
    answers = []
    for start in range(0, len(sample), BATCH_SIZE):
        batch = sample["label"].iloc[start:start + BATCH_SIZE].tolist()
        answers.extend(ask_claude(batch))

    sample["claude"] = answers
    sample.to_csv(sample_path, index=False)

sample["claude_correct"] = sample["claude"] == sample["category"]

print(f"Model accuracy on these labels:  {sample['correct'].mean():.1%}")
print(f"Claude accuracy on these labels: {sample['claude_correct'].mean():.1%}")
print(f"Invalid answers from Claude:     {sample['claude'].isna().sum()}")

Loaded saved Claude answers (no API calls).
Model accuracy on these labels:  49.5%
Claude accuracy on these labels: 88.0%
Invalid answers from Claude:     0


In [6]:
# Compare model and claude at each confidence level 

bins = [0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
sample["confidence_range"] = pd.cut(sample["confidence"], bins=bins)

comparison = sample.groupby("confidence_range", observed=True).agg(
    labels=("label", "size"),
    model_accuracy=("correct", "mean"),
    claude_accuracy=("claude_correct", "mean"),
)
comparison.style.format({"model_accuracy": "{:.0%}", "claude_accuracy": "{:.0%}"})

,labels,model_accuracy,claude_accuracy
confidence_range,,,
"(0.0, 0.4]",100,25%,83%
"(0.4, 0.5]",27,48%,89%
"(0.5, 0.6]",21,52%,100%
"(0.6, 0.7]",12,83%,92%
"(0.7, 0.8]",15,100%,100%
"(0.8, 0.9]",25,100%,88%


In [7]:
# Claude's errors 

claude_errors = sample[~sample["claude_correct"]]
print(f"{len(claude_errors)} Claude errors out of {len(sample)}\n")

claude_errors[["label", "category", "claude", "predicted", "confidence"]].sort_values("category")

24 Claude errors out of 200



,label,category,claude,predicted,confidence
33,CB G20 31/07,Groceries,Restaurants,Shopping,0.358260
79,PAIEMENT CB 03/02 MONTPELLIER G20,Groceries,Shopping,Shopping,0.279015
178,PAIEMENT CB 11/01 BOULOGNE-BILL SMATIS,Health,Shopping,Shopping,0.307173
104,VIR CITYA 20/05,Housing & Utilities,Transfers,Transfers,0.394254
175,PRLV SEPA LYDIC,Housing & Utilities,Transfers,Housing & Utilities,0.457837
165,VIREMENT EN FAVEUR DE SERGIC,Housing & Utilities,Transfers,Transfers,0.348102
91,VIR CNIEG REF FR51ZZZ905348,Income,Housing & Utilities,Income,0.842225
56,PRLV SEPA ON AIR FITNESS FR66ZZZ734595,Leisure & Travel,Health,Leisure & Travel,0.855564
173,CARTE 17/05 NINTENDO ANGERS,Leisure & Travel,Shopping,Shopping,0.221386
151,PRLV SEPA LE CLUB DE FITNESS,Leisure & Travel,Health,Leisure & Travel,0.890865


In [8]:
# Measure the new prompt on the same 200 labels 

v2_path = Path("../data/processed/claude_sample_v2.csv")
BATCH_SIZE = 25

if v2_path.exists():
    v2 = pd.read_csv(v2_path)
    print("Loaded saved answers (no API calls).")
else:
    answers = []
    for start in range(0, len(sample), BATCH_SIZE):
        batch = sample["label"].iloc[start:start + BATCH_SIZE].tolist()
        answers.extend(ask_claude(batch))
    v2 = pd.DataFrame({"claude_v2": answers})
    v2.to_csv(v2_path, index=False)

sample["claude_v2"] = v2["claude_v2"]
sample["claude_v2_correct"] = sample["claude_v2"] == sample["category"]

print(f"Claude, first prompt:  {sample['claude_correct'].mean():.1%}")
print(f"Claude, with rules:    {sample['claude_v2_correct'].mean():.1%}\n")

comparison = sample.groupby("confidence_range", observed=True).agg(
    labels=("label", "size"),
    model=("correct", "mean"),
    claude_v1=("claude_correct", "mean"),
    claude_v2=("claude_v2_correct", "mean"),
)
comparison.style.format({"model": "{:.0%}", "claude_v1": "{:.0%}", "claude_v2": "{:.0%}"})

Loaded saved answers (no API calls).
Claude, first prompt:  88.0%
Claude, with rules:    88.0%



,labels,model,claude_v1,claude_v2
confidence_range,,,,
"(0.0, 0.4]",100,25%,83%,82%
"(0.4, 0.5]",27,48%,89%,89%
"(0.5, 0.6]",21,52%,100%,100%
"(0.6, 0.7]",12,83%,92%,92%
"(0.7, 0.8]",15,100%,100%,93%
"(0.8, 0.9]",25,100%,88%,96%


In [9]:
# Which answers changed ? 

changed = sample[sample["claude_correct"] != sample["claude_v2_correct"]]
print(f"{len(changed)} labels changed between the two prompts\n")

changed[["label", "category", "claude", "claude_v2", "confidence"]].sort_values("confidence")

12 labels changed between the two prompts



,label,category,claude,claude_v2,confidence
169,CB EASYJET STRASBOURG 12/03,Leisure & Travel,Leisure & Travel,Transport,0.204019
173,CARTE 17/05 NINTENDO ANGERS,Leisure & Travel,Shopping,Leisure & Travel,0.221386
30,CARTE 18/04 NATURALIA ROUEN,Groceries,Groceries,Shopping,0.361185
28,PRELEVEMENT GAUMONT REF FR12ZZZ959272,Leisure & Travel,Leisure & Travel,Subscriptions,0.399355
50,PRLV L EQUIPE 16/12,Subscriptions,Leisure & Travel,Subscriptions,0.399630
109,CB AIRBNB ANGERS 11/02,Leisure & Travel,Leisure & Travel,Housing & Utilities,0.404628
1,VIR INST VERS IMMO DE FRANCE,Housing & Utilities,Housing & Utilities,Transfers,0.460183
132,PRLV SEPA XBOX GAME PASS FR57ZZZ215149,Leisure & Travel,Subscriptions,Leisure & Travel,0.491315
180,PRLV SEPA XBOX GAME PASS,Leisure & Travel,Subscriptions,Leisure & Travel,0.491315
21,PRLV SEPA MACIF MUTUELLE,Health,Health,Housing & Utilities,0.727296


In [10]:
# The final evaluation 

from pathlib import Path
from dotenv import load_dotenv
from src.classifier import categorize

load_dotenv("../.env")

for name, fit_data, eval_data in [
    ("synthetic test", train, test),
    ("realistic set", df, realistic),
]:
    path = Path(f"../data/processed/system_{name.replace(' ', '_')}.csv")
    if path.exists():
        output = pd.read_csv(path)
    else:
        classifier = TransactionClassifier(use_embeddings=True).fit(fit_data["label"], fit_data["category"])
        output = pd.DataFrame(categorize(classifier, eval_data["label"]))
        output["truth"] = eval_data["category"].values
        output.to_csv(path, index=False)

    print(f"--- {name} ---")
    evaluate(output["truth"], output["model_category"], "Hybrid alone")
    evaluate(output["truth"], output["category"], "Hybrid + Claude")
    print(f"Below threshold: {(output['confidence'] < 0.7).mean():.0%} | answered by Claude: {(output['source'] == 'claude').mean():.0%}\n")

--- synthetic test ---
Hybrid alone                     accuracy: 54.7% | macro F1: 57.4%
Hybrid + Claude                  accuracy: 96.1% | macro F1: 95.9%
Below threshold: 67% | answered by Claude: 67%

--- realistic set ---
Hybrid alone                     accuracy: 71.7% | macro F1: 71.0%
Hybrid + Claude                  accuracy: 95.0% | macro F1: 95.1%
Below threshold: 60% | answered by Claude: 60%



In [11]:
# Model alone vs Claude alone vs the system

from src.classifier import ask_claude

system = pd.read_csv("../data/processed/system_realistic_set.csv")

claude_only_path = Path("../data/processed/claude_only_realistic.csv")
if claude_only_path.exists():
    claude_only = pd.read_csv(claude_only_path)
else:
    answers = []
    for start in range(0, len(realistic), 25):
        answers.extend(ask_claude(realistic["label"].iloc[start:start + 25].tolist()))
    claude_only = pd.DataFrame({"claude_only": answers})
    claude_only.to_csv(claude_only_path, index=False)

system["claude_only"] = claude_only["claude_only"].fillna("none")

evaluate(system["truth"], system["model_category"], "Hybrid alone")
evaluate(system["truth"], system["claude_only"], "Claude alone")
evaluate(system["truth"], system["category"], "Hybrid + Claude")

errors = system[system["category"] != system["truth"]]
print(f"\nRemaining errors of the system: {len(errors)}\n")
errors[["label", "truth", "category", "source", "confidence"]]

Hybrid alone                     accuracy: 71.7% | macro F1: 71.0%
Claude alone                     accuracy: 95.0% | macro F1: 94.8%
Hybrid + Claude                  accuracy: 95.0% | macro F1: 95.1%

Remaining errors of the system: 3



,label,truth,category,source,confidence
31,PRLV SEPA ALAN SA,Health,Housing & Utilities,claude,0.274723
37,PRLV SEPA L ORANGE BLEUE,Leisure & Travel,Housing & Utilities,claude,0.340141
46,VIR SEPA RECU DE MME MARTIN CLAIRE REMBOURSEME...,Transfers,Income,model,0.909655


In [12]:
# Light model + Claude vs hybrid + Claude

from pathlib import Path
from dotenv import load_dotenv
from src.classifier import categorize

load_dotenv("../.env")

for name, fit_data, eval_data in [
    ("synthetic test", train, test),
    ("realistic set", df, realistic),
]:
    file_name = name.replace(" ", "_")
    light_path = Path(f"../data/processed/system_{file_name}_light.csv")

    if light_path.exists():
        light = pd.read_csv(light_path)
    else:
        classifier = TransactionClassifier(use_embeddings=False).fit(fit_data["label"], fit_data["category"])
        light = pd.DataFrame(categorize(classifier, eval_data["label"]))
        light["truth"] = eval_data["category"].values
        light.to_csv(light_path, index=False)

    hybrid = pd.read_csv(f"../data/processed/system_{file_name}.csv")

    print(f"--- {name} ---")
    evaluate(hybrid["truth"], hybrid["category"], "Hybrid + Claude")
    evaluate(light["truth"], light["category"], "Light + Claude")
    print(f"Answered by Claude: hybrid {(hybrid['source'] == 'claude').mean():.0%} | light {(light['source'] == 'claude').mean():.0%}\n")

--- synthetic test ---
Hybrid + Claude                  accuracy: 96.1% | macro F1: 95.9%
Light + Claude                   accuracy: 96.2% | macro F1: 96.2%
Answered by Claude: hybrid 67% | light 76%

--- realistic set ---
Hybrid + Claude                  accuracy: 95.0% | macro F1: 95.1%
Light + Claude                   accuracy: 96.7% | macro F1: 96.6%
Answered by Claude: hybrid 60% | light 77%

